# Getting Started: Mapping Comparison for H2

This notebook compares several **fermion-to-qubit mappings** for the hydrogen
molecule **H2** using the packaged **VQE** workflow.

Goals:

- run the same VQE setup under different mappings
- compare qubit counts and Hamiltonian sizes
- compare final energies
- inspect the returned convergence traces

This is the basic mapping-comparison workflow in the repository.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from common.hamiltonian import get_exact_spectrum
from vqe.core import run_vqe_mapping_comparison

## Which mappings?

We compare three standard fermion-to-qubit mappings:

- `jordan_wigner`
- `bravyi_kitaev`
- `parity`

These mappings change the qubit representation of the electronic Hamiltonian,
while the underlying physical molecular problem stays the same.

In [ ]:
molecule = "H2"
mappings = ["jordan_wigner", "bravyi_kitaev", "parity"]

ansatz_name = "UCCSD"
optimizer_name = "Adam"
steps = 60
stepsize = 0.2
seed = 0

print("Molecule :", molecule)
print("Mappings :", mappings)
print("Ansatz   :", ansatz_name)
print("Optimizer:", optimizer_name)

## Exact reference energies

The physical spectrum should agree across mappings up to numerical effects, so
it is useful to inspect the exact ground-state energy for each representation.

In [ ]:
exact_ground_energies = {}

for mapping in mappings:
    spectrum = np.asarray(get_exact_spectrum(molecule, mapping=mapping), dtype=float)
    spectrum = np.sort(spectrum)
    exact_ground_energies[mapping] = float(spectrum[0])

print("Exact ground-state energies by mapping:")
for mapping in mappings:
    print(f"{mapping:>15} : {exact_ground_energies[mapping]:.10f} Ha")

## Run the packaged mapping comparison

We keep the variational setup fixed and only change the mapping.

In [ ]:
results = run_vqe_mapping_comparison(
    molecule=molecule,
    ansatz_name=ansatz_name,
    optimizer_name=optimizer_name,
    mappings=mappings,
    steps=steps,
    stepsize=stepsize,
    noisy=False,
    force=True,
    show=True,
    seed=int(seed),
)

In [ ]:
results.keys()

## Summary table

We compare:

- qubit count
- number of Pauli terms, when returned
- final VQE energy
- absolute error relative to the exact ground-state energy for that mapping

In [ ]:
print(f"\n{molecule} VQE — Mapping Comparison Summary\n")

header = (
    f"{'Mapping':>15}  "
    f"{'Qubits':>6}  "
    f"{'Pauli terms':>12}  "
    f"{'E_final (Ha)':>14}  "
    f"{'|ΔE|':>12}"
)
print(header)
print("-" * len(header))

for mapping in mappings:
    data = results[mapping]

    n_qubits = int(data["num_qubits"])
    n_terms = data.get("num_terms", None)
    e_final = float(data["final_energy"])
    e_exact = float(exact_ground_energies[mapping])
    abs_err = abs(e_final - e_exact)

    n_terms_str = str(n_terms) if n_terms is not None else "N/A"

    print(
        f"{mapping:>15}  "
        f"{n_qubits:6d}  "
        f"{n_terms_str:>12}  "
        f"{e_final:14.8f}  "
        f"{abs_err:12.6e}"
    )

## Collect convergence traces

The returned result dictionary stores the VQE traces for each mapping. We plot
them together to compare optimization behavior.

In [ ]:
plt.figure(figsize=(8, 4))

for mapping in mappings:
    data = results[mapping]

    trace = data.get("energies", data.get("energy_trace", None))
    if trace is None:
        continue

    trace = np.asarray(trace, dtype=float)
    plt.plot(np.arange(len(trace)), trace, marker="o", label=mapping)

plt.xlabel("Iteration")
plt.ylabel("Energy [Ha]")
plt.title(f"{molecule} VQE convergence across mappings")
plt.grid(True)
plt.legend()
plt.show()

## Final energies only

A compact plot is useful when the main goal is to compare final performance.

In [ ]:
final_energies = [float(results[m]["final_energy"]) for m in mappings]
exact_energies = [float(exact_ground_energies[m]) for m in mappings]

x = np.arange(len(mappings))

plt.figure(figsize=(8, 4))
plt.plot(x, final_energies, marker="o", label="VQE final energy")
plt.plot(x, exact_energies, marker="o", label="Exact ground energy")
plt.xticks(x, mappings, rotation=20)
plt.ylabel("Energy [Ha]")
plt.title(f"{molecule} final energies across mappings")
plt.grid(True)
plt.legend()
plt.show()

## Interpretation

The mapping changes the qubit Hamiltonian representation, which can affect:

- operator structure
- number of Pauli terms
- optimization behavior

But it does **not** change the underlying molecular physics. In exact arithmetic,
the ground-state energy should represent the same physical quantity across all
mappings.

In practice, a mapping comparison is useful because two mappings can agree on
the target physics while still differing in:

- Hamiltonian sparsity
- circuit structure
- numerical behavior during optimization

## What this notebook showed

We:

- compared `jordan_wigner`, `bravyi_kitaev`, and `parity` for `H2`
- ran the same VQE setup under each mapping
- compared qubit counts, Hamiltonian sizes, and final energies
- inspected the convergence traces

This is the basic mapping-comparison workflow in the repository.

## Next steps

Good follow-ups are:

- repeat the comparison for another molecule
- compare mappings under noisy simulation
- compare mappings with different ansätze
- use the best-performing mapping in a larger bond-scan study